# Import embeddings into pgvector and evaluate val images

Each step is a function in `utils_db`. Postgres must be running (Docker `petrosains-pg`).

Catalog and query embeddings both use **OpenCLIP ViT-B-32** (`laion2b_s34b_b79k`, 512-d). Run `embed_train_official.py` first, then recreate the table here.


## 1. Imports


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "train_embeddings.parquet").exists() and (HERE / "test_pg" / "train_embeddings.parquet").exists():
    HERE = HERE / "test_pg"
REPO = HERE.parent if (HERE.parent / "utils").exists() else HERE
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from utils.embed import load_clip_model
from utils_db import (
    PgConfig,
    apply_vote_rule,
    close_db,
    collect_split_votes,
    connect_db,
    create_hnsw_index,
    create_object_table,
    embed_split_crops,
    evaluate_rule,
    insert_embeddings,
    load_embeddings_parquet,
    load_query_objects,
    load_val_test_splits,
    plot_confusion_matrix,
    save_query_objects,
)


## 2. Database config


In [ ]:
cfg = PgConfig(
    host="127.0.0.1",
    port=5432,
    db="petrosains",
    user="postgres",
    password="ai_squad",
    table="object_embeddings",
    vector_dim=512,
    parquet_path=HERE / "train_embeddings.parquet",
)
cfg


## 3. Load `train_embeddings.parquet`


In [ ]:
df = load_embeddings_parquet(cfg.parquet_path, cfg.vector_dim)
df.head()


## 4. Connect to Postgres


In [ ]:
conn = connect_db(cfg)


## 5. Create `vector` extension and table


In [ ]:
create_object_table(conn, cfg, drop=True)


## 6. Insert parquet rows into pgvector


In [ ]:
insert_embeddings(conn, df, cfg)


## 7. Create HNSW cosine index


In [ ]:
create_hnsw_index(conn, cfg)


## 8. Load val and test images/labels


In [ ]:
splits = load_val_test_splits("dataset")


## 9. Load OpenCLIP ViT-B-32 (run once)


In [ ]:
# Same OpenCLIP ViT-B-32 as embed_train_official.py — run once, reuse below
model = load_clip_model()
print(model.arch, model.pretrained, "on", model.device)


## 10. Save or load val embeddings

First time: run the **save** cell (embeds every val crop, then writes `val_objects.pkl`). Later sessions: skip save and run **load** only.


In [ ]:
val_objects = embed_split_crops(splits, model, split="val")
val_objects = embed_split_crops(splits, model, split="val")
save_query_objects(val_objects, HERE / "val_objects.pkl")


In [ ]:
val_objects = load_query_objects(HERE / "val_objects.pkl")
len(val_objects)
len(val_objects)


## 11. Rank every val crop once (top 100), then apply Rule 3: soft vote (`sum(sim) = avg × n`)

Re-run this cell to build `vote_df`. Later rule cells reuse it and do not query the database again.


In [ ]:
vote_df = collect_split_votes(conn, val_objects, cfg, top_n=100)
pred_df = apply_vote_rule(vote_df, rule="soft")
pred_df


## 12. Confusion matrix for Rule 3 (soft vote), with overall accuracy / precision / recall


In [ ]:
cm = plot_confusion_matrix(
    pred_df,
    splits=splits,
    show=True,
    title="Rule 3: soft vote  sum(sim) = avg × n",
)
cm


## 13. Rule 1: support threshold (`n >= min_n`, then argmax average cosine)


In [ ]:
pred_support, cm_support = evaluate_rule(
    vote_df,
    splits=splits,
    rule="support",
    min_n=3,
    title="Rule 1: support filter  n >= 3, then argmax avg cosine",
)
pred_support


## 14. Rule 2: log-frequency weighted score (`avg × ln(1 + n)`)


In [ ]:
pred_log, cm_log = evaluate_rule(
    vote_df,
    splits=splits,
    rule="log",
    title="Rule 2: log-frequency  avg × ln(1 + n)",
)
pred_log


## 15. Rule 4: Bayesian shrinkage / damped average


In [ ]:
pred_bayes, cm_bayes = evaluate_rule(
    vote_df,
    splits=splits,
    rule="bayes",
    m=3,
    mu0=0.60,
    title="Rule 4: Bayesian shrinkage  (n·avg + m·μ0) / (n + m)",
)
pred_bayes


## 16. Close the database connection


In [ ]:
close_db(conn)


In [ ]:
python predict_labeled_image.py --image L001_test_tubes__L001_TEST_TUBE_L001_04.jpg --label L001_test_tubes__L001_TEST_TUBE_L001_04.txt --output overlay.png